# Random Forest Classification

This notebook trains the Random Forest classifier for the binary stock-direction task. It is aligned with the shared report assumptions:

- `cleaned_dataset.csv` is the single source of truth for all models.
- The positive class is defined as `Stock_Return > 0`.
- Training uses `2014-2017`, and testing uses `2018`.
- The same 201 shared numeric features used by the other models are retained.


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
DATA_PATH = "cleaned_dataset.csv"
sns.set_theme(style="whitegrid")


In [6]:
df = pd.read_csv(DATA_PATH)

excluded_columns = {"Ticker", "Sector", "Stock_Return", "Class", "Year", "Return_Direction"}
feature_columns = [column for column in df.columns if column not in excluded_columns]

y_direction = (df["Stock_Return"] > 0).astype(int)
if "Return_Direction" in df.columns:
    assert (y_direction == df["Return_Direction"]).all(), "Return_Direction does not match Stock_Return > 0."

train_mask = df["Year"].between(2014, 2017)
test_mask = df["Year"] == 2018

X_train = df.loc[train_mask, feature_columns].copy()
X_test = df.loc[test_mask, feature_columns].copy()
y_train = y_direction.loc[train_mask].copy()
y_test = y_direction.loc[test_mask].copy()
train_years = df.loc[train_mask, "Year"].reset_index(drop=True)

print(f"Dataset shape: {df.shape}")
print(f"Number of shared features: {len(feature_columns)}")
print(f"Train rows: {len(X_train):,} | Test rows: {len(X_test):,}")
print(f"Train positive rate: {y_train.mean():.3f}")
print(f"Test positive rate: {y_test.mean():.3f}")


Dataset shape: (22031, 207)
Number of shared features: 201
Train rows: 17,641 | Test rows: 4,390
Train positive rate: 0.513
Test positive rate: 0.692


In [7]:
def make_expanding_year_splits(year_series: pd.Series):
    unique_years = sorted(year_series.unique())
    splits = []

    for index in range(1, len(unique_years)):
        train_year_subset = unique_years[:index]
        validation_year = unique_years[index]

        train_idx = np.where(year_series.isin(train_year_subset))[0]
        validation_idx = np.where(year_series == validation_year)[0]
        splits.append((train_idx, validation_idx))

    return splits


cv_splits = make_expanding_year_splits(train_years)
[(len(train_idx), len(val_idx)) for train_idx, val_idx in cv_splits]


[(3787, 4114), (7901, 4783), (12684, 4957)]

In [8]:
rf_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    (
        "classifier",
        RandomForestClassifier(
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=1,
        ),
    ),
])

param_grid = {
    "classifier__n_estimators": [100, 300],
    "classifier__max_depth": [10, None],
    "classifier__min_samples_split": [2, 10],
}

rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv_splits,
    n_jobs=1,
    verbose=1,
)

rf_grid.fit(X_train, y_train)
best_rf = rf_grid.best_estimator_
best_rf


Fitting 3 folds for each of 8 candidates, totalling 24 fits


KeyboardInterrupt: 

In [ ]:
rf_prob_positive = best_rf.predict_proba(X_test)[:, 1]
rf_pred = best_rf.predict(X_test)

rf_results = pd.Series({
    "accuracy": accuracy_score(y_test, rf_pred),
    "roc_auc": roc_auc_score(y_test, rf_prob_positive),
    "precision": precision_score(y_test, rf_pred, zero_division=0),
    "recall": recall_score(y_test, rf_pred, zero_division=0),
    "f1": f1_score(y_test, rf_pred, zero_division=0),
    "macro_f1": f1_score(y_test, rf_pred, average="macro", zero_division=0),
}, name="Random Forest")

print("Best hyperparameters:")
print(rf_grid.best_params_)
print()
print(rf_results.round(4))


In [ ]:
baseline_models = {
    "Majority Class": DummyClassifier(strategy="most_frequent"),
    "Stratified Random": DummyClassifier(strategy="stratified", random_state=RANDOM_STATE),
}

baseline_rows = []
for model_name, model in baseline_models.items():
    model.fit(X_train, y_train)
    baseline_pred = model.predict(X_test)
    baseline_prob = model.predict_proba(X_test)[:, 1]

    baseline_rows.append({
        "model": model_name,
        "accuracy": accuracy_score(y_test, baseline_pred),
        "roc_auc": roc_auc_score(y_test, baseline_prob),
        "precision": precision_score(y_test, baseline_pred, zero_division=0),
        "recall": recall_score(y_test, baseline_pred, zero_division=0),
        "f1": f1_score(y_test, baseline_pred, zero_division=0),
        "macro_f1": f1_score(y_test, baseline_pred, average="macro", zero_division=0),
    })

results_table = pd.concat([
    pd.DataFrame(baseline_rows).set_index("model"),
    rf_results.to_frame().T,
]).round(4)

results_table


In [ ]:
print(classification_report(y_test, rf_pred, target_names=["Negative return", "Positive return"], zero_division=0))

cm = confusion_matrix(y_test, rf_pred)
cm_df = pd.DataFrame(cm, index=["Actual negative", "Actual positive"], columns=["Predicted negative", "Predicted positive"])
cm_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0])
axes[0].set_title("Random Forest Confusion Matrix")
axes[0].set_xlabel("Predicted label")
axes[0].set_ylabel("True label")

fpr, tpr, _ = roc_curve(y_test, rf_prob_positive)
axes[1].plot(fpr, tpr, label=f"Random Forest (AUC = {roc_auc_score(y_test, rf_prob_positive):.3f})", linewidth=2)
axes[1].plot([0, 1], [0, 1], linestyle="--", color="grey", label="No-skill baseline")
axes[1].set_title("ROC Curve")
axes[1].set_xlabel("False positive rate")
axes[1].set_ylabel("True positive rate")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
feature_importances = pd.Series(
    best_rf.named_steps["classifier"].feature_importances_,
    index=feature_columns,
).sort_values(ascending=False)

top_features = feature_importances.head(20).sort_values()

plt.figure(figsize=(10, 8))
plt.barh(top_features.index, top_features.values, color="#2a9d8f")
plt.title("Top 20 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

top_features.to_frame("importance").sort_values("importance", ascending=False)


## Notes for the report

- The positive class is always `1 = positive future return`, so the reported probability is the probability of an upward return.
- The train/test split exactly matches the report draft: train on `2014-2017`, test on `2018`.
- The feature matrix excludes `Ticker`, `Sector`, `Stock_Return`, `Class`, `Year`, and `Return_Direction`, leaving the shared 201 numeric features.
- The same cleaned CSV can therefore be used consistently across Lasso, Ridge, Random Forest, and XGBoost.
